# 실험 설계 실습

**Design of Experiments · DOE · 실험계획법**

적은 실험으로 변수 효과와 상호작용을 확인할 수 있게 조건을 미리 배치하는 방법.

소재 분야에서 이해하기: 한 번에 한 변수만 바꾸는 대신 요인 설계로 상호작용을 확인한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [SciPy 준몬테카를로 문서](https://docs.scipy.org/doc/scipy/reference/stats.qmc.html)

## 1. OAT와 요인 설계 비교

같은 실험 횟수로 얻는 정보량이 다릅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def response(x1, x2):
    """상호작용이 있는 가상 수율(%): 온도와 시간이 함께 높아야 좋습니다."""
    return 40 + 8 * x1 + 3 * x2 + 9 * x1 * x2

print('꼭짓점 네 곳의 수율:', [round(response(a, b), 1) for a, b in [(-1, -1), (-1, 1), (1, -1), (1, 1)]])

In [ ]:
from sklearn.linear_model import LinearRegression

oat = np.array([[0, 0], [-1, 0], [1, 0], [0, -1], [0, 1]], float)
factorial = np.array([[-1, -1], [-1, 1], [1, -1], [1, 1], [0, 0]], float)

def fit_with_interaction(design):
    y_design = response(design[:, 0], design[:, 1]) + rng.normal(0, 0.3, len(design))
    features = np.column_stack([design, design[:, 0] * design[:, 1]])
    model = LinearRegression().fit(features, y_design)
    return model.coef_

print('참 계수            x1 8.0  x2 3.0  x1*x2 9.0')
print('OAT 설계 추정      x1 %.1f  x2 %.1f  x1*x2 %.1f' % tuple(fit_with_interaction(oat)))
print('요인 설계 추정     x1 %.1f  x2 %.1f  x1*x2 %.1f' % tuple(fit_with_interaction(factorial)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True, sharey=True)
for axis, (name, design) in zip(axes, [('one-at-a-time', oat), ('factorial', factorial)]):
    axis.scatter(design[:, 0], design[:, 1], s=60)
    axis.set_title(name); axis.set_xlabel('temperature (coded)'); axis.grid(alpha=0.3)
axes[0].set_ylabel('time (coded)')
plt.tight_layout(); plt.show()
print('OAT는 중심선 위에만 점이 있어 상호작용 항을 추정할 정보가 없습니다.')

## 2. 해석

요인 설계는 꼭짓점을 씀으로써 같은 실험 횟수로 주효과와 상호작용을 함께 추정합니다.
변수가 많으면 부분 요인 설계나 라틴 하이퍼큐브로 실험 수를 줄입니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#design-of-experiments)을 여세요.